# Stopping-criterion trade-off explorer

Data: `explore_stopmode_curves.csv` (top90 trajectories; `pct_satisfiers` marks the share of the
population that already satisfies every training rule at each generation).

For a stopping threshold `t` (stop the first time `t%` of the population are satisfiers) we read, per
run, the **generation** at which `t` is first reached (the *cost*) and the **best-so-far accuracy** at
that point (what an early-stopped run would return). Sweeping `t` in steps gives a smooth
accuracy-vs-cost curve, averaged over all configurations (optimizer x %rules x repetition).

**Controls**: choose the metric, the threshold step, the x axis (cost or threshold), and *paired*
(restrict to runs that reach the highest threshold, removing survivorship bias at the strict end).
The `highlight %` slider marks a candidate criterion and prints its accuracy / cost / reachability.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

plt.rcParams.update({'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
                     'legend.fontsize': 11})

OUT_DIR = 'figures_stopmode'
NET_LABEL = {'bypass': 'bypass', 'nhlv1': 'NHLy'}
NET_STYLE = {'bypass': ('-', '#1f4e79'), 'nhlv1': (':', '#b5651d')}

def _resolve(p):
    for base in ['', '..', '../..']:
        c = os.path.join(base, p) if base else p
        if os.path.exists(c):
            return c
    return p

def load_runs():
    d = pd.read_csv(_resolve('example/explore_stopmode_curves.csv'))
    d = d[d['stop_mode'] == 'top90']
    runs = {}
    for (net, opt, pct, rep), g in d.groupby(['network', 'optimizer', 'pct', 'rep']):
        g = g.sort_values('generation')
        runs.setdefault(net, []).append({
            'opt': opt, 'pct': int(pct), 'rep': int(rep),
            'gen':  g['generation'].values.astype(float),
            # cumulative max: once t% satisfiers is reached, the threshold stays 'achieved'
            'psat': np.maximum.accumulate(g['pct_satisfiers'].values),
            # best-so-far: what an early-stopped run would actually return
            'bsf':  np.maximum.accumulate(g['acc_best'].values),
            'mean': g['acc_mean'].values.astype(float),
        })
    return runs

RUNS = load_runs()
PCTS = sorted({r['pct'] for v in RUNS.values() for r in v})   # available % of training rules
print('runs per network:', {k: len(v) for k, v in RUNS.items()}, '| % rules:', PCTS)

def reaches(run, t):
    return np.searchsorted(run['psat'], t, side='left') < len(run['psat'])

def at_threshold(run, t, metric):
    idx = np.searchsorted(run['psat'], t, side='left')
    if idx >= len(run['psat']):
        return None
    return run['gen'][idx], (run['bsf'][idx] if metric == 'best-so-far' else run['mean'][idx])

def pool_of(net, pcts=None):
    pool = RUNS[net]
    if pcts is not None:
        pool = [r for r in pool if r['pct'] in pcts]
    return pool

def aggregate(net, thresholds, metric, paired, pcts=None):
    pool = pool_of(net, pcts)
    if paired:
        tmax = max(thresholds)
        pool = [r for r in pool if reaches(r, tmax)]
    acc, cost, reach = [], [], []
    for t in thresholds:
        gv, vv = [], []
        for r in pool:
            res = at_threshold(r, t, metric)
            if res is not None:
                gv.append(res[0]); vv.append(res[1])
        reach.append(len(vv) / len(pool) if pool else np.nan)
        acc.append(np.mean(vv) if vv else np.nan)
        cost.append(np.mean(gv) if gv else np.nan)
    return np.array(acc), np.array(cost), np.array(reach), len(pool)

In [ ]:
w_nets   = W.SelectMultiple(description='networks', options=list(RUNS), value=tuple(RUNS), rows=2)
w_pcts   = W.SelectMultiple(description='% rules', options=PCTS, value=tuple(PCTS), rows=len(PCTS))
w_metric = W.Dropdown(description='metric', options=['best-so-far', 'population-mean'])
w_xaxis  = W.Dropdown(description='x axis', options=['cost (generations)', 'threshold (%)'])
w_tmin   = W.IntSlider(description='min %', value=10, min=2, max=90, step=1, continuous_update=False)
w_step   = W.IntSlider(description='step %', value=10, min=2, max=25, step=1, continuous_update=False)
w_minreach = W.IntSlider(description='min reach %', value=60, min=0, max=100, step=5, continuous_update=False)
w_paired = W.Checkbox(description='paired (runs reaching max threshold)', value=False)
w_hi     = W.IntSlider(description='highlight %', value=50, min=10, max=100, step=5, continuous_update=False)
w_yr     = W.FloatRangeSlider(description='Y', value=[82, 97], min=0, max=100, step=1, continuous_update=False)
w_fname  = W.Text(description='File', placeholder='auto')
w_png    = W.Button(description='Export PNG', button_style='success', icon='download')
out = W.Output(); msg = W.Output()
state = {'fig': None}

def full_reach(net, thresholds, pcts=None):
    pool = pool_of(net, pcts)
    return np.array([np.mean([reaches(r, t) for r in pool]) if pool else np.nan
                     for t in thresholds])

def draw(*_):
    with out:
        out.clear_output(wait=True)
        pcts = set(w_pcts.value) or None
        thr = list(range(w_tmin.value, 101, w_step.value))
        if thr[-1] != 100:
            thr.append(100)
        thr = np.array(thr)
        use_cost = w_xaxis.value.startswith('cost')
        fig, ax = plt.subplots(figsize=(9.5, 5.8))
        for net in [n for n in RUNS if n in w_nets.value]:
            acc, cost, reach, n = aggregate(net, thr, w_metric.value, w_paired.value, pcts)
            # drop thresholds not reliably reachable over the selected runs of this network
            keep = full_reach(net, thr, pcts) >= w_minreach.value / 100.0
            ls, col = NET_STYLE.get(net, ('-', None))
            x = (cost if use_cost else thr)
            ax.plot(x[keep], acc[keep], ls, color=col, lw=2.2, marker='o', ms=4,
                    label=f'{NET_LABEL.get(net, net)} (n={n})')
            hi = w_hi.value
            ai = np.interp(hi, thr, acc)
            ci = np.interp(hi, thr, cost)
            ri = np.interp(hi, thr, reach)
            xi = ci if use_cost else hi
            ax.plot(xi, ai, '*', color=col, ms=18, markeredgecolor='k', zorder=5)
            ax.annotate(f'top{hi}: {ai:.1f}%, gen {ci:.0f}, reached {ri*100:.0f}%',
                        (xi, ai), textcoords='offset points', xytext=(8, -14),
                        fontsize=9, color=col)
        pct_lbl = 'all' if pcts is None or set(pcts) == set(PCTS) else ','.join(map(str, sorted(pcts)))
        ax.set_xlabel('Generations to reach threshold  (cost)' if use_cost
                      else 'Stopping threshold  (% satisfiers)')
        ax.set_ylabel(f'{w_metric.value} rule accuracy (%)')
        ax.set_ylim(w_yr.value); ax.grid(alpha=.3); ax.legend()
        ax.set_title(f'Accuracy vs {"cost" if use_cost else "threshold"} '
                     f'(rules={pct_lbl}%, step {w_step.value}%{", paired" if w_paired.value else ""})',
                     fontweight='bold')
        plt.tight_layout(); state['fig'] = fig; plt.show(); plt.close(fig)

for w in [w_nets, w_pcts, w_metric, w_xaxis, w_tmin, w_step, w_minreach, w_paired, w_hi, w_yr]:
    w.observe(draw, 'value')

def export(_):
    with msg:
        msg.clear_output(wait=True)
        if state['fig'] is None:
            print('Draw first.'); return
        os.makedirs(OUT_DIR, exist_ok=True)
        name = (w_fname.value.strip() or 'stopmode_tradeoff')
        name = name if name.lower().endswith('.png') else name + '.png'
        p = os.path.join(OUT_DIR, name)
        state['fig'].savefig(p, dpi=150, bbox_inches='tight')
        print('Saved:', os.path.abspath(p))
w_png.on_click(export)

ui = W.VBox([W.HBox([w_nets, w_pcts, w_metric, w_xaxis]),
             W.HBox([w_tmin, w_step, w_minreach, w_paired]),
             W.HBox([w_hi, w_yr]),
             W.HBox([w_fname, w_png]), msg, out])
display(ui); draw()